In [23]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
# Make sure Rescaling is in your imports
from tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPooling2D, Rescaling 
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import os
import zipfile

# 1. Download and extract the dataset
_URL = 'https://storage.googleapis.com/mledu-datasets/cats_and_dogs_filtered.zip'
path_to_zip = tf.keras.utils.get_file('cats_and_dogs.zip', origin=_URL, extract=False)

with zipfile.ZipFile(path_to_zip, 'r') as zip_ref:
    zip_ref.extractall(os.path.dirname(path_to_zip))

base_dir = os.path.join(os.path.dirname(path_to_zip), 'cats_and_dogs_filtered')
train_dir = os.path.join(base_dir, 'train')
validation_dir = os.path.join(base_dir, 'validation')

# 2. Create the datasets using the correct settings
BATCH_SIZE = 32
# --- THIS IS THE FIX ---
# Define IMG_SIZE as a tuple (height, width)
IMG_SIZE = (150, 150) 
# ----------------------

ds_train = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    shuffle=True,
    image_size=IMG_SIZE, # This now correctly receives the tuple
    batch_size=BATCH_SIZE
)

ds_test = tf.keras.utils.image_dataset_from_directory(
    validation_dir,
    shuffle=False,
    image_size=IMG_SIZE, # This also receives the tuple
    batch_size=BATCH_SIZE
)

print("Data loading complete.")

Found 2000 files belonging to 2 classes.
Found 1000 files belonging to 2 classes.
Data loading complete.


In [24]:
# Create the ANN model with Rescaling layer
ann_model_catsdogs = Sequential([
    # Add the Rescaling layer to normalize pixel values from [0, 255] to [0, 1]
    Rescaling(1./255, input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
    
    Flatten(), # No need to specify input_shape here anymore
    Dense(128, activation='relu'),
    Dense(1, activation='sigmoid')
])

# Print the updated model summary
print("Updated ANN Architecture:")
ann_model_catsdogs.summary()

Updated ANN Architecture:


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ rescaling_1 (Rescaling)              │ (None, 150, 150, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_5 (Flatten)                  │ (None, 67500)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_10 (Dense)                     │ (None, 128)                 │       8,640,128 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_11 (Dense)                     │ (None, 1)                   │             129 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 8,640,257 (32.96 MB)

 Trainable params: 8,640,257 (32.96 MB)

 Non-trainable params: 0 (0.00 B)

In [26]:
# Create the CNN model with the Rescaling layer
cnn_model_catsdogs = Sequential([
    # Add the Rescaling layer to normalize and fix input shape
    Rescaling(1./255, input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
    
    # First convolutional block (no longer needs input_shape)
    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),

    # Second convolutional block
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),

    # Flatten the results to feed into a dense layer
    Flatten(),
    Dense(128, activation='relu'),
    # Output layer for binary classification
    Dense(1, activation='sigmoid')
])

# Print the updated model summary
print("\nUpdated CNN Architecture:")
cnn_model_catsdogs.summary()


Updated CNN Architecture:


Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ rescaling_2 (Rescaling)              │ (None, 150, 150, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_6 (Conv2D)                    │ (None, 148, 148, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_6 (MaxPooling2D)       │ (None, 74, 74, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_7 (Conv2D)                    │ (None, 72, 72, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_7 (MaxPooling2D)       │ (None, 36, 36, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_7 (Flatten)                  │ (None, 82944)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_14 (Dense)                     │ (None, 128)                 │      10,616,960 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_15 (Dense)                     │ (None, 1)                   │             129 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 10,636,481 (40.57 MB)

 Trainable params: 10,636,481 (40.57 MB)

 Non-trainable params: 0 (0.00 B)

In [27]:
# Callbacks for the ANN model
ann_cd_callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, verbose=1),
    ModelCheckpoint('best_ann_model_catsdogs.keras', save_best_only=True, monitor='val_accuracy', mode='max')
]

# Callbacks for the CNN model
cnn_cd_callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, verbose=1),
    ModelCheckpoint('best_cnn_model_catsdogs.keras', save_best_only=True, monitor='val_accuracy', mode='max')
]

In [28]:
# Compile the ANN model
ann_model_catsdogs.compile(optimizer='adam',
                           loss='binary_crossentropy',
                           metrics=['accuracy'])

# Train the ANN model
print("\n--- Training ANN for Cats vs Dogs ---")
ann_cd_history = ann_model_catsdogs.fit(ds_train,
                                        epochs=10,
                                        validation_data=ds_test,
                                        callbacks=ann_cd_callbacks)


--- Training ANN for Cats vs Dogs ---
Epoch 1/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 23s 341ms/step - accuracy: 0.5158 - loss: 9.2214 - val_accuracy: 0.5480 - val_loss: 1.4337
Epoch 2/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 30s 160ms/step - accuracy: 0.5647 - loss: 1.9791 - val_accuracy: 0.5470 - val_loss: 1.9262
Epoch 3/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 13s 205ms/step - accuracy: 0.5829 - loss: 1.4523 - val_accuracy: 0.5060 - val_loss: 3.8388
Epoch 4/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 12s 189ms/step - accuracy: 0.5666 - loss: 2.1298 - val_accuracy: 0.5590 - val_loss: 0.9946
Epoch 5/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 10s 164ms/step - accuracy: 0.6028 - loss: 1.1760 - val_accuracy: 0.5730 - val_loss: 0.9281
Epoch 6/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 10s 150ms/step - accuracy: 0.5852 - loss: 1.4025 - val_accuracy: 0.5070 - val_loss: 3.1621
Epoch 7/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 11s 154ms/step - accuracy: 0.5627 - loss: 2.4835 - val_accuracy: 0.5080 - val_loss: 4.2949
Epoch 8/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 10s 152ms/step - acc

In [29]:
# Compile the CNN model
cnn_model_catsdogs.compile(optimizer='adam',
                           loss='binary_crossentropy',
                           metrics=['accuracy'])

# Train the CNN model
print("\n--- Training CNN for Cats vs Dogs ---")
cnn_cd_history = cnn_model_catsdogs.fit(ds_train,
                                        epochs=10,
                                        validation_data=ds_test,
                                        callbacks=cnn_cd_callbacks)


--- Training CNN for Cats vs Dogs ---
Epoch 1/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 38s 574ms/step - accuracy: 0.4995 - loss: 1.2364 - val_accuracy: 0.5350 - val_loss: 0.6925
Epoch 2/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 36s 570ms/step - accuracy: 0.5494 - loss: 0.6902 - val_accuracy: 0.6550 - val_loss: 0.6493
Epoch 3/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 36s 573ms/step - accuracy: 0.6512 - loss: 0.6369 - val_accuracy: 0.5780 - val_loss: 0.9209
Epoch 4/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 41s 567ms/step - accuracy: 0.7040 - loss: 0.5930 - val_accuracy: 0.6830 - val_loss: 0.5948
Epoch 5/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 36s 574ms/step - accuracy: 0.8094 - loss: 0.4406 - val_accuracy: 0.6890 - val_loss: 0.6392
Epoch 6/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 36s 574ms/step - accuracy: 0.8415 - loss: 0.3459 - val_accuracy: 0.6900 - val_loss: 0.7369
Epoch 7/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 36s 567ms/step - accuracy: 0.9055 - loss: 0.2538 - val_accuracy: 0.6880 - val_loss: 0.7208
Epoch 7: early stopping


In [30]:
# Load the best models
best_ann_cd = tf.keras.models.load_model('best_ann_model_catsdogs.keras')
best_cnn_cd = tf.keras.models.load_model('best_cnn_model_catsdogs.keras')

# Evaluate the ANN
print("\n--- Evaluating ANN on Test Data ---")
ann_cd_loss, ann_cd_accuracy = best_ann_cd.evaluate(ds_test)
print(f"ANN Test Accuracy: {ann_cd_accuracy * 100:.2f}%")

# Evaluate the CNN
print("\n--- Evaluating CNN on Test Data ---")
cnn_cd_loss, cnn_cd_accuracy = best_cnn_cd.evaluate(ds_test)
print(f"CNN Test Accuracy: {cnn_cd_accuracy * 100:.2f}%")


--- Evaluating ANN on Test Data ---
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step - accuracy: 0.6556 - loss: 0.7371
ANN Test Accuracy: 57.30%

--- Evaluating CNN on Test Data ---
32/32 ━━━━━━━━━━━━━━━━━━━━ 4s 110ms/step - accuracy: 0.7885 - loss: 0.5159
CNN Test Accuracy: 69.00%
